In [1]:
import re
import os
from dotenv import load_dotenv

from atlassian import Confluence
from markdownify import markdownify as md
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from typing import List, Dict

load_dotenv()

# ========================== CONFIG ==========================
CONFLUENCE_PAGE_ID = 1671169
COLLECTION_NAME = "python_coding_standards"
DB_PATH = "./chroma_python_standards_db2"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# ========================== FETCH FROM CONFLUENCE ==========================
def fetch_confluence_page(page_id: int):
    confluence = Confluence(
        url=os.getenv("CF_URL"),
        username=os.getenv("ACCOUNT"),
        password=os.getenv("CF_TOKEN")
    )

    page = confluence.get_page_by_id(page_id=page_id, expand='body.storage')
    html_content = page['body']['storage']['value']
    title = page.get('title', 'Python Coding Standards')

    print(f"✅ Fetched page: '{title}' (ID: {page_id})")
    return html_content, title

In [2]:
# ========================== CLEAN HTML → BETTER MARKDOWN ==========================
def html_to_clean_markdown(html_content: str, page_title: str) -> str:
    # Improved markdownify settings for Confluence
    markdown = md(
        html_content,
        heading_style="ATX",
        bullets="*",
        code_language="python",           # Default for code blocks
        strip=['img', 'script', 'style'],
        convert_links=True,
        escape_underscores=False,         # Important: prevent \_ escaping
        escape_asterisks=False
    )

    # === Post-cleaning steps to remove noises ===

    # 1. Remove "wide760", "pywide760", and similar Confluence width artifacts
    markdown = re.sub(r'wide760|pywide760|wide \d+', '', markdown, flags=re.IGNORECASE)

    # 2. Fix broken code block markers and escaped characters
    markdown = re.sub(r'\\([*_`])', r'\1', markdown)   # Remove unnecessary escapes: \_ → _
    markdown = re.sub(r'```python\n+```', '', markdown)  # Remove empty code blocks

    # 3. Clean up excessive newlines and spaces
    markdown = re.sub(r'\n{4,}', '\n\n\n', markdown)   # Max 3 blank lines
    markdown = re.sub(r'^\s+$', '', markdown, flags=re.MULTILINE)

    # 4. Fix common table and list issues
    markdown = re.sub(r'\\\|', '|', markdown)           # Fix escaped pipes in tables

    # 5. Ensure ASCII diagrams stay in code blocks (optional: wrap loose diagrams)
    # If diagram starts with +--- or |, wrap it in ```ascii
    def wrap_ascii_diagrams(text):
        lines = text.split('\n')
        in_diagram = False
        result = []
        for line in lines:
            if re.match(r'^\s*[\+\|]', line.strip()) and not in_diagram:
                result.append('```ascii')
                in_diagram = True
            elif in_diagram and not line.strip():
                result.append('```')
                in_diagram = False
            result.append(line)
        if in_diagram:
            result.append('```')
        return '\n'.join(result)

    markdown = wrap_ascii_diagrams(markdown)

    # Final markdown with title
    final_md = f"# {page_title}\n\n{markdown.strip()}"
    
    print("✅ Converted to clean Markdown (noise removed)")
    return final_md


In [3]:
# # ========================== HEADER-BASED CHUNKING (Same as before) ==========================
# def chunk_by_headers(markdown_text: str) -> List[Dict]:
#     header_pattern = re.compile(r'^(#{1,4})\s+(.*)', re.MULTILINE)
    
#     chunks = []
#     current_title = "Document Root"
#     current_level = 0
#     current_content = []
#     last_pos = 0

#     for match in header_pattern.finditer(markdown_text):
#         start = match.start()
#         level = len(match.group(1))
#         title = match.group(2).strip()

#         if current_content:
#             content = "\n".join(current_content).strip()
#             if content:
#                 chunks.append({
#                     "chunk_id": f"chunk_{len(chunks)}",
#                     "section_title": current_title,
#                     "header_level": current_level,
#                     "content": content,
#                     "parent_section": chunks[-1]["section_title"] if chunks else "None",
#                     "source": "Confluence_Python_Coding_Standards"
#                 })

#         current_title = title
#         current_level = level
#         current_content = [match.group(0)]
#         last_pos = match.end()

#     # Last chunk
#     current_content.append(markdown_text[last_pos:])
#     content = "\n".join(current_content).strip()
#     if content:
#         chunks.append({
#             "chunk_id": f"chunk_{len(chunks)}",
#             "section_title": current_title,
#             "header_level": current_level,
#             "content": content,
#             "parent_section": chunks[-1]["section_title"] if chunks else "None",
#             "source": "Confluence_Python_Coding_Standards"
#         })

#     return chunks


In [4]:
def chunk_by_headers(markdown_text: str) -> List[Dict]:
    header_pattern = re.compile(r'^(#{1,4})\s+(.*)', re.MULTILINE)

    chunks = []
    matches = list(header_pattern.finditer(markdown_text))

    for i, match in enumerate(matches):
        level = len(match.group(1))
        title = match.group(2).strip()

        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(markdown_text)

        content = markdown_text[start:end].strip()

        chunks.append({
            "chunk_id": f"chunk_{i}",
            "section_title": title,
            "header_level": level,
            "content": content,
            "parent_section": matches[i - 1].group(2).strip() if i > 0 else "None",
            "source": "Confluence_Python_Coding_Standards"
        })

    return chunks

In [5]:
# ========================== INGEST TO CHROMADB ==========================
def ingest_to_chroma(chunks: List[Dict], page_title: str):

    embedding_function = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)

    client = chromadb.PersistentClient(path=DB_PATH)
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_function,
        metadata={"hnsw:space": "cosine"}
    )

    ids = [c["chunk_id"] for c in chunks]
    documents = [c["content"] for c in chunks]
    metadatas = [{
        "section_title": c["section_title"],
        "header_level": c["header_level"],
        "parent_section": c["parent_section"],
        "source": c["source"],
        "page_title": page_title
    } for c in chunks]

    collection.add(ids=ids, documents=documents, metadatas=metadatas)

    print("🚀 Ingestion completed successfully!")


In [6]:
html_content, page_title = fetch_confluence_page(CONFLUENCE_PAGE_ID)

✅ Fetched page: 'Test age 2' (ID: 1671169)


In [7]:
markdown_content = html_to_clean_markdown(html_content, page_title)

✅ Converted to clean Markdown (noise removed)


In [8]:
# Save cleaned markdown for debugging
with open("clean_python_coding_standards2.md", "w", encoding="utf-8") as f:
    f.write(markdown_content)
print("✅ Saved cleaned Markdown to: clean_python_coding_standards2.md")

✅ Saved cleaned Markdown to: clean_python_coding_standards2.md


In [9]:
chunks = chunk_by_headers(markdown_content)
print(f"✅ Created {len(chunks)} clean chunks")

✅ Created 23 clean chunks


In [10]:
print(f"Type of first chunk: {type(chunks[0])}, Keys: {chunks[0].keys()}")

Type of first chunk: <class 'dict'>, Keys: dict_keys(['chunk_id', 'section_title', 'header_level', 'content', 'parent_section', 'source'])


In [11]:
for chunk in chunks:
    # print(f"--- {chunk['section_title']} (Level {chunk['header_level']}) ---")
    # print(chunk['content'][:200] + "...\n")  # Print first 200 chars of content for preview
    print(chunk)  # Print first 200 chars of content for preview
    print("--"*50)

{'chunk_id': 'chunk_0', 'section_title': 'Test age 2', 'header_level': 1, 'content': "Here's the **updated Python Coding Standards** page for Confluence, now using **plain Markdown ASCII flow diagrams** (text-based flowcharts) instead of Mermaid or images.\n\nThese ASCII diagrams render cleanly in Confluence when pasted as code blocks or plain text.\n\n---", 'parent_section': 'None', 'source': 'Confluence_Python_Coding_Standards'}
----------------------------------------------------------------------------------------------------
{'chunk_id': 'chunk_1', 'section_title': 'Python Coding Standards', 'header_level': 1, 'content': '**Document Owner:** Engineering Team   \n**Version:** 1.3   \n**Last Updated:** March 2026\n\n---', 'parent_section': 'Test age 2', 'source': 'Confluence_Python_Coding_Standards'}
----------------------------------------------------------------------------------------------------
{'chunk_id': 'chunk_2', 'section_title': 'Table of Contents', 'header_level': 2, 'co

In [12]:
ingest_to_chroma(chunks, page_title)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Ingestion completed successfully!
